# OOP Week 5 -- Plotter & Reporter Components

**Course:** Object-Oriented Programming (Year 2)
**Session:** 3 hours
**Prerequisites:** Weeks 1-4
**Focus:** component architecture, unified exports, visualization objects

---

## Learning Objectives

1. Build a Plotter class that creates figures from analysis results
2. Build a Reporter class that exports all outputs
3. Understand the full component pipeline: Source -> Clean -> Analyze -> Plot -> Report
4. Design classes that produce file outputs

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/ArifSolmaz/courseos-curriculum.git
# %cd courseos-curriculum/course-content

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Setup: Previous components

In [ ]:
import os, json

class Dataset:
    def __init__(self, rows, source_path="unknown"):
        self.rows = rows
        self.source_path = source_path
        self.n_rows = len(rows)
        self.columns = list(rows[0].keys()) if rows else []
    def get_column(self, name):
        return [row.get(name) for row in self.rows]
    def __len__(self):
        return self.n_rows

# Sample clean data and analysis results
clean_data = Dataset([
    {"id": i, "value": v, "status": "ok"}
    for i, v in enumerate([25, 30, 88, 42, 67, 15, 55, 73, 38, 91], 1)
], "sensors.csv")

sample_results = {
    "analysis_summary": {
        "column": "value",
        "count": 10,
        "mean": 52.4,
        "std": 25.5,
        "min": 15.0,
        "max": 91.0,
    }
}
print("Setup complete. Clean data:", len(clean_data), "rows")

**Expected Output:**
```
Setup complete. Clean data: 10 rows
```

---
## Section 1: The Plotter Component

The Plotter creates visualizations from analysis results. It:
- Takes a Dataset and results dictionary as input
- Creates matplotlib figures
- Saves them to disk
- Returns the list of created figures

Note: If matplotlib is not available, the Plotter gracefully degrades (prints a message instead of crashing).

In [ ]:
class Plotter:
    """Creates and saves figures from analysis results."""

    def __init__(self, config):
        self.config = config
        self.figures = []
        self.saved_paths = []

    def plot(self, dataset, results):
        """Create all required plots."""
        try:
            import matplotlib
            matplotlib.use("Agg")
            import matplotlib.pyplot as plt
        except ImportError:
            print("matplotlib not available -- skipping plots")
            return []

        self.figures = []
        self.saved_paths = []
        fig_dir = self.config.get("figures_dir", "reports/figures")
        os.makedirs(fig_dir, exist_ok=True)

        col = self.config.get("value_column", "value")
        values = dataset.get_column(col)
        values = [v for v in values if isinstance(v, (int, float))]

        if not values:
            print("No numeric values to plot")
            return []

        # Figure 1: Time series
        fig1, ax = plt.subplots(figsize=(10, 4))
        ax.plot(values, color="#2196F3", linewidth=0.8, marker="o", markersize=3)
        title = self.config.get("project_name", "") + " -- Time Series"
        ax.set_title(title)
        ax.set_xlabel("Index")
        ax.set_ylabel(col)
        ax.grid(True, alpha=0.3)
        path1 = os.path.join(fig_dir, "timeseries.png")
        fig1.savefig(path1, dpi=100, bbox_inches="tight")
        self.figures.append(fig1)
        self.saved_paths.append(path1)
        plt.close(fig1)

        # Figure 2: Histogram
        fig2, ax = plt.subplots(figsize=(8, 4))
        ax.hist(values, bins=20, color="#2196F3", edgecolor="white")
        ax.set_title("Distribution of " + col)
        ax.set_xlabel(col)
        ax.set_ylabel("Count")
        ax.grid(True, alpha=0.3)
        path2 = os.path.join(fig_dir, "histogram.png")
        fig2.savefig(path2, dpi=100, bbox_inches="tight")
        self.figures.append(fig2)
        self.saved_paths.append(path2)
        plt.close(fig2)

        print("Created " + str(len(self.figures)) + " figures")
        for p in self.saved_paths:
            print("  Saved: " + p)
        return self.figures


# Test
plotter = Plotter({
    "project_name": "SensorDemo",
    "value_column": "value",
    "figures_dir": "/tmp/oop_demo/figures",
})
figs = plotter.plot(clean_data, sample_results)
print("Figures created:", len(figs))

**Expected Output:**
```
Created 2 figures
  Saved: /tmp/oop_demo/figures/timeseries.png
  Saved: /tmp/oop_demo/figures/histogram.png
Figures created: 2
```

---
## Section 2: The Reporter Component

The Reporter is the final stage. It collects everything and exports it to the required formats (CSV, JSON, etc).

In [ ]:
import csv

class Reporter:
    """Generates and exports reports."""

    def __init__(self, config):
        self.config = config
        self.exported = {}

    def export(self, dataset, results, figure_paths=None):
        """Export all outputs to configured paths."""
        self.exported = {}

        # 1. Export cleaned CSV
        csv_path = self.config.get("cleaned_data_path", "data/cleaned/cleaned.csv")
        os.makedirs(os.path.dirname(csv_path), exist_ok=True)
        if dataset.rows:
            cols = list(dataset.rows[0].keys())
            with open(csv_path, "w", newline="") as f:
                w = csv.DictWriter(f, fieldnames=cols)
                w.writeheader()
                w.writerows(dataset.rows)
            self.exported["cleaned_csv"] = csv_path
            print("Exported CSV: " + csv_path)

        # 2. Export report JSON
        report_path = self.config.get("report_path", "reports/report.json")
        os.makedirs(os.path.dirname(report_path), exist_ok=True)
        report = {
            "project_name": self.config.get("project_name"),
            "track": self.config.get("track"),
            "version": "v3",
            "dataset": {"n_clean": len(dataset)},
            "analysis_summary": results.get("analysis_summary", {}),
            "figures": figure_paths or [],
        }
        with open(report_path, "w") as f:
            json.dump(report, f, indent=2)
        self.exported["report"] = report_path
        print("Exported report: " + report_path)

        print("Total exports: " + str(len(self.exported)))
        return self.exported


# Test
reporter = Reporter({
    "project_name": "SensorDemo",
    "track": "robotics",
    "cleaned_data_path": "/tmp/oop_demo/data/cleaned.csv",
    "report_path": "/tmp/oop_demo/reports/report.json",
})
exports = reporter.export(clean_data, sample_results, plotter.saved_paths)

**Expected Output:**
```
Exported CSV: /tmp/oop_demo/data/cleaned.csv
Exported report: /tmp/oop_demo/reports/report.json
Total exports: 2
```

---
## Section 3: The Full Pipeline

Now let us see all 5 components working together. This is the architecture you will build over the next weeks.

---
### Full v3 Pipeline Architecture

```
DataSource --> Dataset --> Cleaner --> Dataset --> Analyzer --> results
                                                      |          |
                                                      v          v
                                                   Plotter   Reporter
                                                      |          |
                                                  figures/    reports/
                                                   *.png     report.json
                                                             cleaned.csv
```

---
### Try It!

Add a third plot type to the Plotter: a box plot of the values. Hint: use `ax.boxplot(values)`.

In [ ]:
# YOUR CODE HERE


---
## Section 4: More Plot Types

In [ ]:
class EnhancedPlotter(Plotter):
    """Plotter with additional chart types."""

    def plot_boxplot(self, dataset, col):
        """Create a box plot."""
        try:
            import matplotlib
            matplotlib.use('Agg')
            import matplotlib.pyplot as plt
        except ImportError:
            print('matplotlib not available')
            return None

        values = [v for v in dataset.get_column(col)
                  if isinstance(v, (int, float))]
        if not values:
            return None

        fig, ax = plt.subplots(figsize=(6, 4))
        ax.boxplot(values, vert=True)
        ax.set_title('Box Plot of ' + col)
        ax.set_ylabel(col)
        ax.grid(True, alpha=0.3)
        plt.close(fig)
        print('Created box plot for ' + col)
        return fig

    def plot_bar(self, labels, values, title='Bar Chart'):
        """Create a bar chart."""
        try:
            import matplotlib
            matplotlib.use('Agg')
            import matplotlib.pyplot as plt
        except ImportError:
            print('matplotlib not available')
            return None

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(labels, values, color='#2196F3')
        ax.set_title(title)
        ax.grid(True, alpha=0.3, axis='y')
        plt.close(fig)
        print('Created bar chart: ' + title)
        return fig


ep = EnhancedPlotter({
    'project_name': 'Demo',
    'value_column': 'value',
    'figures_dir': '/tmp/oop_demo/figures',
})

# Box plot
box_fig = ep.plot_boxplot(clean_data, 'value')

# Bar chart from analysis results
bar_fig = ep.plot_bar(
    ['Mean', 'Std', 'Min', 'Max'],
    [52.4, 25.5, 15.0, 91.0],
    'Analysis Summary'
)

**Expected Output:**
```
Created box plot for value
Created bar chart: Analysis Summary
```

---
## Section 5: Reporter Formats

In [ ]:
class EnhancedReporter(Reporter):
    """Reporter with multiple output formats."""

    def export_text_summary(self, results, path):
        """Export a human-readable text summary."""
        os.makedirs(os.path.dirname(path), exist_ok=True)
        lines = []
        lines.append('=== Analysis Report ===')
        lines.append('')
        summary = results.get('analysis_summary', {})
        for key, val in summary.items():
            if isinstance(val, float):
                lines.append(key + ': ' + str(round(val, 4)))
            else:
                lines.append(key + ': ' + str(val))
        text = chr(10).join(lines)
        with open(path, 'w') as f:
            f.write(text)
        print('Exported text summary: ' + path)
        return path


er = EnhancedReporter({
    'project_name': 'Demo',
    'track': 'data',
    'cleaned_data_path': '/tmp/oop_demo/data/cleaned.csv',
    'report_path': '/tmp/oop_demo/reports/report.json',
})
er.export_text_summary(sample_results, '/tmp/oop_demo/reports/summary.txt')

**Expected Output:**
```
Exported text summary: /tmp/oop_demo/reports/summary.txt
```

---
### Procedural vs OOP: Report Generation

The Reporter class tracks its own state (what was exported), can be configured once and used repeatedly, and can be extended with new formats via subclassing (OCP).

**Procedural approach (what you did in CP1/CP2):**

In [ ]:
# PROCEDURAL
def export_report(data, results, csv_path, json_path):
    # Save CSV
    import csv
    with open(csv_path, 'w') as f:
        w = csv.DictWriter(f, fieldnames=data[0].keys())
        w.writeheader()
        w.writerows(data)
    # Save JSON
    import json
    with open(json_path, 'w') as f:
        json.dump(results, f)
    # Add a new format? Modify this function!
    return {'csv': csv_path, 'json': json_path}

**OOP approach (what we are learning now):**

In [ ]:
# OOP
reporter = Reporter(config)
exports = reporter.export(dataset, results, figures)
# reporter.exported tracks what was created
# Adding new format = add new method or subclass
# No existing code changes needed

---
### Design Decision: Graceful degradation for optional dependencies

Notice how Plotter wraps matplotlib in try/except. This is **graceful degradation** -- the program still works without matplotlib, it just skips the plots.

This is important because:
- Not every environment has matplotlib installed
- Tests should run without GUI dependencies
- Server environments often lack display capabilities

**Pattern:** Wrap optional imports in try/except at the point of use, not at the top of the file.

---
### Try It!

Add a `export_markdown(self, results, path)` method to EnhancedReporter that outputs the analysis as a Markdown table. Example output:
```
| Metric | Value |
|--------|-------|
| mean   | 52.4  |
| std    | 25.5  |
```

In [ ]:
# YOUR CODE HERE


---
### Try It!

Modify the full pipeline to take a single config dict and run all 5 components automatically. The config should specify data path, cleaning rules, analysis types, and output paths.

In [ ]:
# YOUR CODE HERE


---
## Build from Scratch Exercise

This exercise tests whether you truly understand this week's concepts. Complete it without looking at the examples above.

In [ ]:
# BUILD FROM SCRATCH:
# Build a SimpleReporter that takes analysis results and exports them as both CSV and JSON. Include a text summary method.

# YOUR CODE HERE


In [ ]:
# TEST your build-from-scratch code:

# YOUR TESTS HERE


---
## Connect the Dots

How does this week's concept connect to previous weeks?

In [ ]:
# Draw the full pipeline showing how DataSource, Cleaner, Analyzer, Plotter, and Reporter connect.

# YOUR ANSWER (as comments or code):


---
## Real-World Spotting

OOP patterns are everywhere in real software. Can you spot them?

In [ ]:
# Think about a social media app. What would its Reporter export? (profile data, activity stats, media files?)

# YOUR ANSWER:


---
## Diagram It

Draw an ASCII class diagram for the main classes from this week. Include:
- Class names
- Key attributes
- Key methods
- Relationships (has-a, is-a)

In [ ]:
# Draw your ASCII diagram here:
# +------------------+
# |   ClassName      |
# +------------------+
# | - attribute      |
# +------------------+
# | + method()       |
# +------------------+

# YOUR DIAGRAM:


---
## Key Vocabulary

| Term | Definition |
|------|------------|
| **Component** | A self-contained object with a clear interface |
| **Graceful degradation** | Continuing to work when optional features are missing |
| **Export** | Writing data to external files (CSV, JSON, PNG) |

---
## Recap Exercise

Without looking at the code above, try to:

In [ ]:
# 1. Write one class from this week FROM MEMORY
#    (it does not need to be perfect)

# YOUR CODE HERE


# 2. Create an instance and call at least one method

# YOUR CODE HERE


# 3. Write one test for your class

# YOUR CODE HERE


---
## What to Review Before Next Week

Before the next session, make sure you can:

1. Explain this week's main concept in your own words
2. Write a simple example from memory
3. Identify this pattern in existing code
4. Explain WHY this pattern is useful (not just HOW)

---
## Mini-Quiz

In [ ]:
# Q1: What are the 5 components of the v3 pipeline?
# Answer: 

# Q2: Why does the Plotter use try/except for matplotlib?
# Answer: 

# Q3: What files does the Reporter produce?
# Answer: 

---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)